# ReadmitIQ - Clinical Note Embeddings on Colab GPU

This self-contained notebook embeds the synthetic clinical notes for all three splits (`train`, `val`, `test`) using `all-MiniLM-L6-v2` on a free T4 GPU in ~3-5 minutes.

### Instructions
1. In Google Colab, change runtime to **GPU** (`Runtime > Change runtime type > T4 GPU`).
2. Upload the 3 parquet files to the Colab working directory:
   - `train.parquet`
   - `val.parquet`
   - `test.parquet`
3. Run all cells.
4. Download the generated output files:
   - `embeddings_train.npy`
   - `embeddings_val.npy`
   - `embeddings_test.npy`
   - `embeddings_checksum.json`
5. Place them into `data/processed/` in the local repository.

In [ ]:
!pip install -q sentence-transformers pandas pyarrow

In [ ]:
import hashlib
import json
from pathlib import Path
import time
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using compute device: {device}')
assert device == 'cuda', 'GPU recommended for fast embedding! Go to Runtime > Change runtime type > T4 GPU'

model_name = 'all-MiniLM-L6-v2'
print(f'Loading {model_name} onto {device}...')
model = SentenceTransformer(model_name, device=device)

In [ ]:
def compute_sha256(filepath):
    h = hashlib.sha256()
    with open(filepath, 'rb') as f:
        while chunk := f.read(8192):
            h.update(chunk)
    return h.hexdigest()

splits = ['train', 'val', 'test']
metadata = {}

for split in splits:
    parquet_file = f'{split}.parquet'
    assert Path(parquet_file).exists(), f'Missing {parquet_file}! Please upload it.'
    df = pd.read_parquet(parquet_file)
    assert 'clinical_note' in df.columns, f'clinical_note column missing from {parquet_file}!'
    notes = df['clinical_note'].tolist()
    n_rows = len(notes)
    print(f'[{split.upper()}] Encoding {n_rows} notes with batch_size=256...')
    
    t0 = time.time()
    embeddings = model.encode(
        notes,
        batch_size=256,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype(np.float32)
    elapsed = time.time() - t0
    
    assert embeddings.shape == (n_rows, 384), f'Expected ({n_rows}, 384), got {embeddings.shape}'
    out_file = f'embeddings_{split}.npy'
    np.save(out_file, embeddings)
    print(f'[{split.upper()}] Saved {out_file} in {elapsed:.1f}s.')
    
    metadata[split] = {
        'rows': n_rows,
        'embedding_dim': 384,
        'file': out_file,
        'sha256': compute_sha256(out_file),
        'seconds': round(elapsed, 2)
    }

with open('embeddings_checksum.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print('Encoding complete! Metadata saved to embeddings_checksum.json.')

In [ ]:
# Verification check
with open('embeddings_checksum.json') as f:
    meta = json.load(f)
print(json.dumps(meta, indent=2))

# Zip for one-click download
!zip -j readmit_embeddings.zip embeddings_train.npy embeddings_val.npy embeddings_test.npy embeddings_checksum.json
print('Ready! Download readmit_embeddings.zip from the Colab file browser.')